In [1]:
import numpy as np

In [2]:
import pandas as pd

In [3]:
pd.set_option('display.max_rows',None);
pd.set_option('display.max_columns',None)
train=pd.concat([pd.read_csv('/kaggle/input/datasets/shaadahmad51/qwerty1/SDNFlow_Dataset_Normal.csv'),pd.read_csv('/kaggle/input/datasets/shaadahmad51/qwerty1/SDNFlow_Dataset_attack.csv')])
train.head()
train=train.drop(columns=['flow_ID','eth_type','ipv4_src','ipv4_dst','ip_proto','src_port','dst_port','TnBPDstIP','TnBPSrcIP','TnPPDstIP','TnPPSrcIP','TnPPerDport','Tn_FlowsPDstIP','Tn_FlowsPSrcIP','Attack'])
train.head()
tester=pd.read_csv('/kaggle/input/datasets/shaadahmad51/qwerty1/modified_concatenated_InSDN_DatasetCSV.csv')
tester.head()
tester = tester.drop(columns=[
    'Pkt Len Min',
    'Pkt Len Max',
    'Pkt Len Mean',
    'Pkt Len Std',
    'Pkt Len Var',
    'FIN Flag Cnt',
    'SYN Flag Cnt',
    'RST Flag Cnt',
    'PSH Flag Cnt',
    'ACK Flag Cnt',
    'URG Flag Cnt',
    'CWE Flag Count',
    'ECE Flag Cnt',
    'Down/Up Ratio'
])
tester = tester.drop(columns=[
    'Flow IAT Mean',
    'Flow IAT Std',
    'Flow IAT Max',
    'Flow IAT Min',
    'Fwd IAT Tot',
    'Fwd IAT Mean',
    'Fwd IAT Std',
    'Fwd IAT Max',
    'Fwd IAT Min',
    'Bwd IAT Tot',
    'Bwd IAT Mean',
    'Bwd IAT Std',
    'Bwd IAT Max',
    'Bwd IAT Min',
    'Fwd PSH Flags',
    'Bwd PSH Flags',
    'Fwd URG Flags',
    'Bwd URG Flags',
    'Fwd Header Len',
    'Bwd Header Len'
])
tester = tester.drop(columns=[
    'Fwd Pkt Len Max',
    'Fwd Pkt Len Min',
    'Fwd Pkt Len Mean',
    'Fwd Pkt Len Std',
    'Bwd Pkt Len Max',
    'Bwd Pkt Len Min',
    'Bwd Pkt Len Mean',
    'Bwd Pkt Len Std'
])
tester=tester.drop(columns=['Flow ID','Timestamp','Flow Duration'])

tester['Tot Fwd Pkts']=tester['Tot Fwd Pkts']+tester['Tot Bwd Pkts']
tester=tester.rename(columns={'Tot Fwd Pkts':'tot_pkts'})

tester['TotLen Fwd Pkts']=tester['TotLen Fwd Pkts']+tester['TotLen Bwd Pkts']
tester=tester.rename(columns={'TotLen Fwd Pkts':'tot_len_pkts'})

tester=tester.drop(columns=['Tot Bwd Pkts','TotLen Bwd Pkts','Fwd Pkts/s','Bwd Pkts/s'])

tester=tester.rename(columns={'Flow Byts/s':'flow_byts_s','Flow Pkts/s':'flow_pkts_s','Pkt Size Avg':'pkt_size_average', 'Active Mean': 'active_mean',
    'Active Std': 'active_std',
    'Active Max': 'active_max',
    'Active Min': 'active_min',
    'Idle Mean': 'idle_mean',
    'Idle Std': 'idle_std',
    'Idle Max': 'idle_max',
    'Idle Min': 'idle_min'
})
train = train.drop(columns=[
    'fwd_header_len',
    'Byts_max_interval_2s',
    'Byts_min_interval_2s',
    'Byts_mean_interval_2s',
    'Byts_std_interval_2s',
    'Pkts_max_interval_2s',
    'Pkts_mean_interval_2s',
    'Pkts_min_interval_2s',
    'Pkts_std_interval_2s'
])

In [4]:

train=train.drop(columns='flow_duration')
train.head()
y=train['Category']
train.head()
X=train
y.head()
y=y.str.upper()
y.head()
y.value_counts()
y=y[~y.isin(['STREAMING','SSH','ICMP','FTP','NTP'])]
y.value_counts()
tester['Label'].value_counts()
tester['Label']=tester['Label'].str.upper()
tester['Label'].value_counts()
tester['Label']=tester['Label'].str.strip()
tester['Label'].value_counts()

y=y.replace({
    'HTTP':'NORMAL',
    'PASSWORD-GUESSING':'BFA',
    'DNS':'BOTNET'
})
y.value_counts()
y.isnull().sum()
X.isnull().sum()
from sklearn.model_selection import train_test_split
X.shape
y.shape
y.head()
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
y=le.fit_transform(y)
X.shape
y.shape
train['Category'].value_counts()
safe_train=train
train['Category']=train['Category'].str.strip()
train['Category']=train['Category'].str.upper()
train['Category'].value_counts()
train=train[~train['Category'].isin(['STREAMING','SSH','ICMP','FTP','NTP'])]
train['Category'].value_counts()
y=train['Category']
y=le.fit_transform(y)
df=pd.DataFrame(y)
df.value_counts()


X=train.drop(columns='Category')
X.shape
y.shape
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=2)
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)

In [5]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader, TensorDataset
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# ==========================================
# GAN ARCHITECTURE
# ==========================================
class Generator(nn.Module):
    def __init__(self, latent_dim, data_dim):
        super(Generator, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, data_dim)
        )
    def forward(self, z):
        return self.model(z)

class Discriminator(nn.Module):
    def __init__(self, data_dim):
        super(Discriminator, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(data_dim, 128),
            nn.LeakyReLU(0.2),
            nn.Linear(128, 64),
            nn.LeakyReLU(0.2),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.model(x)

# ==========================================
# OVERSAMPLING & METRICS PIPELINE
# ==========================================
def balance_and_evaluate_gan(X_train, y_train, X_test, y_test, latent_dim=10, epochs=100, batch_size=32, lr=0.0001,beta1=0.1):
    """
    Oversamples minority classes (0-7) using GANs and evaluates with a Classification Report.
    """
    # 1. Evaluate Benchmark (Before GAN)
    print("="*60)
    print(" EVALUATION 1: ORIGINAL PRE-PROCESSED DATA (IMBALANCED)")
    print("="*60)
    clf_orig = RandomForestClassifier(random_state=42)
    clf_orig.fit(X_train, y_train)
    preds_orig = clf_orig.predict(X_test)
    print(classification_report(y_test, preds_orig, zero_division=0))

    # Convert training data to numpy formats for processing
    X_arr = X_train.values if isinstance(X_train, pd.DataFrame) else np.array(X_train)
    y_arr = y_train.values if isinstance(y_train, pd.Series) else np.array(y_train).flatten()
    
    class_counts = pd.Series(y_arr).value_counts()
    majority_count = class_counts.max()
    
    X_balanced = X_arr.copy()
    y_balanced = y_arr.copy()
    all_classes = sorted(list(np.unique(y_arr)))
    
    print("\n" + "="*60)
    print(f" STARTING MULTI-CLASS GAN OVERSAMPLING (Target per class: {majority_count})")
    print("="*60)

    # 2. Train GAN loop for each minority class
    for current_class in all_classes:
        minority_count = class_counts.get(current_class, 0)
        samples_needed = majority_count - minority_count
        
        if samples_needed <= 0:
            print(f"-> Class {current_class}: Already majority ({minority_count} samples). Skipping.")
            continue
            
        print(f"-> Class {current_class}: Training GAN to generate {samples_needed} synthetic rows...")
        
        # Isolate rows belonging to this class
        minority_data = X_arr[y_arr == current_class]
        data_dim = minority_data.shape[1]
        
        tensor_x = torch.tensor(minority_data, dtype=torch.float32)
        dataloader = DataLoader(TensorDataset(tensor_x), batch_size=min(batch_size, len(minority_data)), shuffle=True)
        
        # Networks and Optimizers
        gen = Generator(latent_dim, data_dim)
        disc = Discriminator(data_dim)
        loss_fn = nn.BCELoss()
        opt_G = torch.optim.Adam(gen.parameters(), lr=lr, betas=(0.5, 0.999))
        opt_D = torch.optim.Adam(disc.parameters(), lr=lr, betas=(0.5, 0.999))
        
        # Training Routine
        gen.train()
        disc.train()
        for epoch in range(epochs):
            for (real_samples,) in dataloader:
                b_size = real_samples.size(0)
                real_lbl = torch.ones(b_size, 1)
                fake_lbl = torch.zeros(b_size, 1)
                
                # Discriminator step
                opt_D.zero_grad()
                d_real = loss_fn(disc(real_samples), real_lbl)
                z = torch.randn(b_size, latent_dim)
                fake_samples = gen(z)
                d_fake = loss_fn(disc(fake_samples.detach()), fake_lbl)
                (d_real + d_fake).backward()
                opt_D.step()
                
                # Generator step
                opt_G.zero_grad()
                g_loss = loss_fn(disc(fake_samples), real_lbl)
                g_loss.backward()
                opt_G.step()
                
        # Generate samples for this class
        gen.eval()
        with torch.no_grad():
            z_final = torch.randn(samples_needed, latent_dim)
            synthetic_features = gen(z_final).numpy()
            
        synthetic_labels = np.full((samples_needed,), current_class)
        
        # Merge data
        X_balanced = np.vstack((X_balanced, synthetic_features))
        y_balanced = np.concatenate((y_balanced, synthetic_labels))

    # Re-wrap to Pandas DataFrame/Series structure if inputs were Pandas
    if isinstance(X_train, pd.DataFrame):
        X_balanced = pd.DataFrame(X_balanced, columns=X_train.columns)
    if isinstance(y_train, pd.Series):
        y_balanced = pd.Series(y_balanced, name=y_train.name)

    # 3. Evaluate Balanced Output (After GAN)
    print("\n" + "="*60)
    print(" EVALUATION 2: BALANCED DATA (POST GAN OVERSAMPLING)")
    print("="*60)
    clf_bal = RandomForestClassifier(n_estimators=300, random_state=42,max_depth=None,min_samples_split=2,min_samples_leaf=1)
    clf_bal.fit(X_balanced, y_balanced)
    preds_bal = clf_bal.predict(X_test)
    print(classification_report(y_test, preds_bal, zero_division=0))
    
    return X_balanced, y_balanced

# ==========================================
# HOW TO TRIGGER THE PIPELINE
# ==========================================
# Paste your pre-existing variables inside this function call:
X_train_balanced, y_train_balanced = balance_and_evaluate_gan(X_train, y_train, X_test, y_test)


 EVALUATION 1: ORIGINAL PRE-PROCESSED DATA (IMBALANCED)
              precision    recall  f1-score   support

           0       0.94      0.27      0.42     14920
           1       0.89      0.24      0.38     10696
           2       0.93      0.90      0.92     24665
           3       0.91      0.69      0.78     32805
           4       0.94      0.79      0.86     13452
           5       0.48      0.97      0.65     30574
           6       0.91      0.77      0.83      3573
           7       0.94      0.81      0.87      1273

    accuracy                           0.72    131958
   macro avg       0.87      0.68      0.71    131958
weighted avg       0.82      0.72      0.71    131958


 STARTING MULTI-CLASS GAN OVERSAMPLING (Target per class: 132169)
-> Class 0: Training GAN to generate 72445 synthetic rows...
-> Class 1: Training GAN to generate 89314 synthetic rows...
-> Class 2: Training GAN to generate 33027 synthetic rows...
-> Class 3: Already majority (132169 sample